# 04 · Backtest
Model predictions on the test period -> long/flat signals -> vectorbt, vs buy & hold.

In [ ]:
import sys; sys.path.insert(0, '../src')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from quant_dl.data import download
from quant_dl.features import FEATURE_COLUMNS, add_indicators, make_windows, chronological_split
from quant_dl.models import LSTMModel
from quant_dl.backtest import run_backtest, signals_from_predictions

In [ ]:
ckpt = torch.load('../checkpoints/lstm_aapl.pt', weights_only=False)
WINDOW = ckpt['window']
df = add_indicators(download('AAPL', '2018-01-01', '2025-01-01')).dropna()
X, y = make_windows(df, FEATURE_COLUMNS, WINDOW)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = chronological_split(X, y)
X_test_s = (X_test - ckpt['mu']) / ckpt['sigma']

In [ ]:
model = LSTMModel(n_features=len(FEATURE_COLUMNS))
model.load_state_dict(ckpt['state_dict'])
model.eval()
with torch.no_grad():
    pred = model(torch.from_numpy(X_test_s)).numpy()

In [ ]:
# test-period close prices, aligned with predictions
test_close = df['close'].iloc[-len(X_test):]
entries, exits = signals_from_predictions(test_close, pred, threshold=0.0)
stats = run_backtest(test_close, entries, exits)
pd.Series(stats)

In [ ]:
bh = run_backtest(test_close,
                  pd.Series(True, index=test_close.index),
                  pd.Series(False, index=test_close.index))
compare = pd.DataFrame({'strategy': stats, 'buy_and_hold': bh})
compare

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
(test_close / test_close.iloc[0]).plot(ax=ax, label='buy & hold')
strat_ret = (1 + test_close.pct_change().fillna(0) * entries.shift(1).fillna(False)).cumprod()
strat_ret.plot(ax=ax, label='LSTM strategy')
ax.legend(); ax.set_title('Cumulative return (test period)'); plt.show()